# Shopper simulation: demo

A small agent-based model of 500 synthetic shoppers choosing between six yoghurts over 26 weeks.
A new product, *Live Kefir*, launches in week 4. Every parameter is an **assumption**, not calibrated
to data, so read the results as *what the model implies*, not as predictions about real shoppers.

Run the cells from top to bottom (from the `notebooks/` folder).

In [ ]:
import sys
sys.path.insert(0, "../src")          # use the code in src/ without installing the package

import pandas as pd
from abm.config import SimConfig
from abm.metrics import compare_scenarios, repeat_metrics
from abm.plots import (plot_market_share, plot_trial_by_scenario,
                       plot_share_change, plot_sensitivity)
from abm.scenarios import default_scenarios
from abm.sensitivity import (KEFIR, sweep, kefir_repeat_rate, kefir_trial_rate,
                             brand_price_rise_effect)
from abm.simulate import run_simulation

cfg = SimConfig()                      # 500 agents, 26 weeks, seed 42
scenarios = default_scenarios()
results = {name: run_simulation(cfg, sc) for name, sc in scenarios.items()}

## 1. What happens with no outside events (baseline)

Each line is a product's share of that week's purchases. The kefir launches in week 4 and takes share
from every product. Weekly shares wobble by a few points because only about 250 shoppers buy in a given
week: that is **Monte Carlo noise**, not a trend.

In [ ]:
plot_market_share(results['baseline']);

## 2. Scenarios: same shoppers, same random draws, different outside events

All scenarios use the same seed, so they are **identical until the event starts** (the lines below sit on
top of each other until week 8) and any gap afterwards is caused by the event. This is the
*common random numbers* trick.

- `kefir_promo`: 25% off the kefir in weeks 8-11
- `brand_price_rise`: Classic Brand +25% from week 12
- `health_trend`: everyone cares 50% more about healthiness from week 10

In [ ]:
plot_trial_by_scenario(results, promo_weeks=(8, 11));

In [ ]:
plot_share_change(compare_scenarios(results), results['baseline']);

The price rise moves Classic Brand's share down by about 5 points over the whole run, and the lost sales
spread across all the others. The health trend helps the healthy products (kefir, Greek protein) at the
expense of the indulgent dessert, as intended.

In [ ]:
rows = {name: repeat_metrics(res, KEFIR) for name, res in results.items()}
pd.DataFrame(rows).T.round(2)

**Trial vs repeat.** The promotion raises *trial* (more people try the kefir) but the *repeat rate* is
unchanged: people who tried it on promotion come back about as often as everyone else. A temporary price
cut brings in new triers, but in this model it doesn't create loyalty.

## 3. Sensitivity: which assumptions drive the answers?

Each panel changes **one** parameter and repeats the run over 15 seeds. The line is the average and the
band is the 95% confidence interval of that average, so bands that don't overlap are real differences.

In [ ]:
seeds = range(15)
habit = [0, 1, 2.5, 5, 8, 12]
repeat_sw = sweep(cfg, "beta.habit", habit, kefir_repeat_rate, seeds)
trial_sw = sweep(cfg, "beta.habit", habit, kefir_trial_rate, seeds)
price_sw = sweep(cfg, "beta.price", [-1, -2, -3, -4.5, -6], brand_price_rise_effect, seeds)
price_sw["value"] = -price_sw["value"]      # plot price sensitivity as a positive number

plot_sensitivity([
    ("Kefir repeat rate vs habit strength", "habit coefficient", repeat_sw, 2.5, True),
    ("Kefir trial rate vs habit strength", "habit coefficient", trial_sw, 2.5, True),
    ("Brand share change after +25% price", "price sensitivity", price_sw, 3.0, False),
]);

**Reading the panels**

- **Habit cuts both ways.** Stronger habit raises the kefir's repeat rate (about 44% to 51%) but sharply
  lowers its trial rate (about 67% to 34%), because shoppers locked into their usual yoghurt don't try the
  new one. For a new-product launch, habit is the barrier to entry *and* the source of retention.
- **At the default habit strength the repeat rate barely depends on habit.** The bands for 0 and 2.5
  overlap, so repeat buying here comes mostly from fixed tastes. Habit strength is the first parameter
  to calibrate against real repeat-purchase data.
- **Price response is not a straight line.** Making shoppers more price-sensitive increases the damage from
  a price rise up to a point, then it levels off: when they are already very price-sensitive, the brand has
  little share left to lose.

## What this model does *not* do

No shopper-to-shopper influence, no stock-outs or shelf placement, independent weekly missions, one habit
speed for everyone, and parameters set by assumption. See `LIMITATIONS.md` (step 8) for how RGC data would
be used to calibrate and validate each of these.